# CAM Intersection Defence — Unilingual Attack (2-mod)

Defends against a **unilingual (English-only) typographic attack** (EN text at both boxes)
using GradCAM intersection masking.

**2-mod**: intersect GradCAM(EN model, EN text) ∩ GradCAM(ZH model, EN-text-via-ZH-encoder)

The ZH model is probed with English class-name text to find where it attends even when the
attack language is foreign to it.

Results saved to `results/cam_2mod/`.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'open_clip_torch', 'transformers', 'datasets',
                'matplotlib', 'Pillow'], check=False)

CompletedProcess(args=['d:\\ian\\2026summer\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'open_clip_torch', 'transformers', 'datasets', 'matplotlib', 'Pillow'], returncode=0)

In [2]:
import importlib, sys, os, platform, random, json, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import ChineseCLIPModel, ChineseCLIPProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

LANGS = ['en', 'zh']
CLASSES = {
    'en': ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck'],
    'zh': ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车'],
}
TMPL = {'en': 'a photo of a {}.', 'zh': '一张{}的照片。'}

RESULTS_DIR  = 'results/cam_2mod'
CACHE_DIR    = os.path.join(RESULTS_DIR, 'cache')
CACHE_DIR_4MOD = CACHE_DIR   # reuse naming convention
os.makedirs(CACHE_DIR, exist_ok=True)

SETUP_LABEL  = 'unilingual'
ATTACK_LABEL = 'unilingual'

Device: cuda


## 1. Model definitions

In [3]:
def classify(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i + batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

def classify_conf(model, imgs, words, batch_size=128):
    """Returns (predictions, max-cosine-sim confidences)."""
    preds, confs = [], []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i + batch_size])
        tf  = model.embed_texts(words)
        sims = imf @ tf.t()
        preds.append(sims.argmax(-1).cpu().numpy())
        confs.append(sims.max(-1).values.cpu().numpy())
    return np.concatenate(preds), np.concatenate(confs)

def _clip_feat(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True, return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'], attention_mask=t['attention_mask'],
                                        token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

MODEL_CLS = {'en': EnCLIP, 'zh': ZhCLIP}
print('Model classes defined:', list(MODEL_CLS.keys()))

Model classes defined: ['en', 'zh']


## 2. Attack helpers

In [4]:
DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
_FONT_CACHE  = {}

def _font_paths():
    if platform.system() == 'Windows':
        winfonts = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        cjk   = os.path.join(winfonts, 'msyh.ttc')
        latin = os.path.join(winfonts, 'arial.ttf')
        if not os.path.exists(latin): latin = cjk
        return (cjk if os.path.exists(cjk) else None,
                latin if os.path.exists(latin) else None)
    for d in ['/usr/share/fonts', '/Library/Fonts', os.path.expanduser('~/.fonts')]:
        for f in ['NotoSansCJK-Regular.ttc', 'NotoSans-Regular.ttf']:
            p = os.path.join(d, f)
            if os.path.exists(p): return p, p
    return None, None

_CJK_FONT, _LAT_FONT = _font_paths()

def _font_for(word):
    return _CJK_FONT if any(ord(c) > 127 for c in word) else _LAT_FONT

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:    _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except: _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]



def _draw_text_box(draw, word, rect, font):
    rx, ry, rx2, ry2 = rect
    bb = draw.textbbox((0, 0), word, font=font)
    draw.rectangle([rx, ry, rx2, ry2], fill='white')
    draw.text((rx + PAD - bb[0], ry + PAD - bb[1]), word, fill='black', font=font)

print('Font paths:', _CJK_FONT, _LAT_FONT)


Font paths: C:\WINDOWS\Fonts\msyh.ttc C:\WINDOWS\Fonts\arial.ttf


In [5]:
def _clamp_xy(xy, bw, bh):
    x, y = int(xy[0]), int(xy[1])
    x = max(0, min(x, max(0, DISPLAY_SIZE - bw)))
    y = max(0, min(y, max(0, DISPLAY_SIZE - bh)))
    return x, y

def draw_word(img, word, img_idx, already_224=False):
    """Place the same word at frozen EN/L slot anchors from sample JSON."""
    fp = _font_for(word)
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    font  = _get_font(fp)
    draw  = ImageDraw.Draw(img)
    bb    = draw.textbbox((0, 0), word, font=font)
    bw    = (bb[2] - bb[0]) + 2 * PAD
    bh    = (bb[3] - bb[1]) + PAD + 12
    for xy in (attack_pos['en'][int(img_idx)], attack_pos['l'][int(img_idx)]):
        rx, ry = _clamp_xy(xy, bw, bh)
        _draw_text_box(draw, word, (rx, ry, rx + bw, ry + bh), font)
    return img


def build_attacked_images(base_imgs, img_indices, attack_lang, n_workers=None):
    """Unilingual attack: same word in both boxes."""
    words     = [CLASSES[attack_lang][target[int(k)]] for k in img_indices]
    n_workers = n_workers or min(8, os.cpu_count() or 4)
    def _one(args):
        im, word, img_idx = args
        return draw_word(im, word, img_idx=int(img_idx), already_224=True)
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        return list(pool.map(_one, zip(base_imgs, words, img_indices)))

print(f'Unilingual attack helper ready: {NUM_BOXES} boxes @ size {FONT_SIZE} (both EN)')


Unilingual attack helper ready: 2 boxes @ size 24 (both EN)


## 3. Dataset

In [6]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../../../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
attack_pos = _saved['attack_pos']
assert len(attack_pos['en']) == len(idx) and len(attack_pos['l']) == len(idx)
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000
assert np.array_equal(true, np.array(_saved['true']))
assert all((true == c).sum() == 100 for c in range(10))

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean     = [im.convert('RGB') for im in rows[image_key]]
print('Upscaling clean images to 224px...', end=' ', flush=True)
t0 = time.time()
clean_224 = [im.resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC) for im in clean]
print(f'{time.time()-t0:.1f}s')

all_idx  = np.arange(len(clean))
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Loaded {len(clean)} images; tune subset = {len(tune_idx)}')


Upscaling clean images to 224px... 0.3s
Loaded 1000 images; tune subset = 100


## 4. Load models + build unilingual attack

In [7]:
models = {}
for lang, cls in MODEL_CLS.items():
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')

# Standard text embeddings: each model with its own language
TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in LANGS}

clean_preds = {lang: classify(models[lang], clean_224, CLASSES[lang]) for lang in LANGS}
clean_acc   = {lang: float((clean_preds[lang] == true).mean()) for lang in LANGS}
print('Clean acc:', {k: f'{100*v:.1f}%' for k, v in clean_acc.items()})

Loading en... 

d:\ian\2026summer\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


2.4s
Loading zh... 

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

2.1s
Clean acc: {'en': '85.9%', 'zh': '91.4%'}


In [8]:
print('Building unilingual EN attacked images...')
t0 = time.time()
attacked_imgs = build_attacked_images(clean_224, all_idx, 'en')
print(f'Done in {time.time()-t0:.1f}s')

preds_attacked = {ml: classify(models[ml], attacked_imgs, CLASSES[ml]) for ml in LANGS}
baseline_acc   = {ml: float((preds_attacked[ml] == true).mean())   for ml in LANGS}
baseline_asr   = {ml: float((preds_attacked[ml] == target).mean()) for ml in LANGS}
print('Baseline acc:', {k: f'{100*v:.1f}%' for k, v in baseline_acc.items()})
print('Baseline ASR:', {k: f'{100*v:.1f}%' for k, v in baseline_asr.items()})

Building unilingual EN attacked images...
Done in 0.2s
Baseline acc: {'en': '3.4%', 'zh': '27.1%'}
Baseline ASR: {'en': '96.5%', 'zh': '71.8%'}


## 5. GradCAM helpers + cross text embedding

For unilingual 2-mod:
1. GradCAM(EN model, EN text)          — standard
2. GradCAM(ZH model, EN-via-ZH text)   — ZH model probed with English class names

In [9]:
def _norm_cam(cam):
    cam = cam.relu() if isinstance(cam, torch.Tensor) else np.maximum(cam, 0)
    cam = cam.detach().cpu().numpy() if isinstance(cam, torch.Tensor) else cam
    cam = cam - cam.min()
    mx  = cam.max()
    return cam / mx if mx > 0 else cam

def _cam_from_conv(act, grad):
    w = grad.mean(dim=(2, 3), keepdim=True)
    return _norm_cam((w * act).sum(dim=1).squeeze(0))

def gradcam_en(pil_img, target_idx):
    wrapper = models['en']; acts = {}
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle = wrapper.m.visual.conv1.register_forward_hook(hook)
    x        = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    feat     = wrapper.m.visual(x)
    img_feat = F.normalize(feat, dim=-1)
    score    = (img_feat @ TEXT_EMB['en'][target_idx:target_idx+1].T).squeeze()
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam

def gradcam_zh(pil_img, target_idx):
    wrapper = models['zh']; acts = {}
    patch  = wrapper.m.vision_model.embeddings.patch_embedding
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle = patch.register_forward_hook(hook)
    pv     = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    out    = wrapper.m.get_image_features(pixel_values=pv)
    img_feat = F.normalize(_clip_feat(out), dim=-1)
    score  = (img_feat @ TEXT_EMB['zh'][target_idx:target_idx+1].T).squeeze()
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam

GRADCAM_FN = {'en': gradcam_en, 'zh': gradcam_zh}
print('Standard GradCAM helpers ready.')

Standard GradCAM helpers ready.


In [10]:
def align_cam(cam, size=DISPLAY_SIZE):
    return np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    ) / 255.0

def n_cam_intersection(*cams):
    """Elementwise min of N CAMs after resizing to DISPLAY_SIZE."""
    return np.minimum.reduce([align_cam(c) for c in cams])

def dilate_mask(mask, iterations=3):
    m = mask.astype(bool)
    for _ in range(iterations):
        pad = np.pad(m, 1, mode='constant', constant_values=False)
        m = (pad[:-2,:-2]|pad[:-2,1:-1]|pad[:-2,2:]|
             pad[1:-1,:-2]|pad[1:-1,1:-1]|pad[1:-1,2:]|
             pad[2:,:-2]  |pad[2:,1:-1]  |pad[2:,2:])
    return m

def cam_to_mask(saliency, threshold=0.85, dilate=3):
    thr  = np.percentile(saliency, threshold * 100)
    mask = saliency >= thr
    if dilate > 0: mask = dilate_mask(mask, iterations=dilate)
    return mask

def apply_mask(pil_img, mask, fill='mean_nonmask'):
    pil_img = pil_img.convert('RGB')
    arr = np.array(pil_img)
    h, w = arr.shape[:2]
    if mask.shape != (h, w):
        mask = np.array(
            Image.fromarray(mask.astype(np.uint8) * 255).resize((w, h), Image.NEAREST)
        ) > 127
    out = arr.copy()
    m   = mask.astype(bool)
    if fill == 'mean_nonmask':
        bg = ~m
        mean_color = arr[bg].mean(0) if bg.any() else arr.reshape(-1, 3).mean(0)
        out[m] = mean_color
    return Image.fromarray(out.astype(np.uint8))

def overlay_cam(pil_img, cam, alpha=0.50):
    cam_img = Image.fromarray((cam * 255).astype(np.uint8)).resize(
        (DISPLAY_SIZE, DISPLAY_SIZE), Image.BILINEAR)
    heat  = cm.jet(np.array(cam_img) / 255.0)[:, :, :3]
    base  = np.array(pil_img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE))).astype(np.float32) / 255.0
    blend = np.clip((1 - alpha) * base + alpha * heat, 0, 1)
    return Image.fromarray((blend * 255).astype(np.uint8))

def mask_overlay(pil_img, mask, alpha=0.45):
    arr = np.array(pil_img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE))).astype(np.float32)
    red = np.zeros_like(arr); red[:, :, 0] = 255
    m   = mask.astype(np.float32)[..., None]
    return Image.fromarray((arr * (1 - alpha * m) + red * (alpha * m)).astype(np.uint8))

print('CAM masking helpers ready.')

CAM masking helpers ready.


In [11]:
# Cross-language text embeddings:
#   TEXT_EMBS[('en','zh')] = EN model's text encoder applied to ZH class names
#   TEXT_EMBS[('zh','en')] = ZH model's text encoder applied to EN class names
# These are "cross-lingual probes" - each model sees the other language's vocabulary.

with torch.no_grad():
    # EN model encoding ZH class names (EN tokenizer processes CJK glyphs as UNK/subwords)
    _t_zh_via_en = models['en'].tok(
        [TMPL['en'].format(w) for w in CLASSES['zh']]
    ).to(DEVICE)
    TEXT_EMB_EN_ZH = F.normalize(models['en'].m.encode_text(_t_zh_via_en), dim=-1).detach()

    # ZH model encoding EN class names
    _t_en_via_zh = models['zh'].p(
        text=[TMPL['zh'].format(w) for w in CLASSES['en']],
        padding=True, return_tensors='pt',
    ).to(DEVICE)
    _out_en_via_zh = models['zh'].m.get_text_features(
        input_ids=_t_en_via_zh['input_ids'],
        attention_mask=_t_en_via_zh['attention_mask'],
        token_type_ids=_t_en_via_zh.get('token_type_ids'),
    )
    TEXT_EMB_ZH_EN = F.normalize(_clip_feat(_out_en_via_zh), dim=-1).detach()

# Unified lookup: (model_lang, text_lang) -> text_embedding
TEXT_EMBS = {
    ('en', 'en'): TEXT_EMB['en'],     # standard EN
    ('en', 'zh'): TEXT_EMB_EN_ZH,    # EN model, ZH class names
    ('zh', 'en'): TEXT_EMB_ZH_EN,    # ZH model, EN class names
    ('zh', 'zh'): TEXT_EMB['zh'],     # standard ZH
}
print('Cross-language text embeddings computed.')

Cross-language text embeddings computed.


In [12]:
def gradcam_en_with_emb(pil_img, text_emb, target_idx=None):
    """GradCAM using EN model, scored against text_emb[target_idx]."""
    wrapper = models['en']; acts = {}
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle   = wrapper.m.visual.conv1.register_forward_hook(hook)
    x        = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    feat     = wrapper.m.visual(x)
    img_feat = F.normalize(feat, dim=-1)
    sims     = (img_feat @ text_emb.T).squeeze(0)
    if target_idx is None:
        target_idx = int(sims.detach().argmax().item())
    score = sims[target_idx]
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam, target_idx

def gradcam_zh_with_emb(pil_img, text_emb, target_idx=None):
    """GradCAM using ZH model, scored against text_emb[target_idx]."""
    wrapper = models['zh']; acts = {}
    patch   = wrapper.m.vision_model.embeddings.patch_embedding
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle   = patch.register_forward_hook(hook)
    pv       = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    out      = wrapper.m.get_image_features(pixel_values=pv)
    img_feat = F.normalize(_clip_feat(out), dim=-1)
    sims     = (img_feat @ text_emb.T).squeeze(0)
    if target_idx is None:
        target_idx = int(sims.detach().argmax().item())
    score = sims[target_idx]
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam, target_idx

def get_n_cams(pil_img, combos):
    """Compute CAMs for all (model_lang, text_lang) combos. Returns list of cams."""
    cams = []
    for ml, tl in combos:
        emb = TEXT_EMBS[(ml, tl)]
        fn  = gradcam_en_with_emb if ml == 'en' else gradcam_zh_with_emb
        cam, _ = fn(pil_img, emb)
        cams.append(cam)
    return cams

print('Generalised GradCAM helpers ready.')

Generalised GradCAM helpers ready.


## 6. CAM cache

In [13]:
# Unilingual 2-mod uses combos: (EN model, EN text) and (ZH model, EN text)
COMBOS_UNI_2MOD = [('en', 'en'), ('zh', 'en')]

def compute_and_cache_cams(condition, indices, combos=COMBOS_UNI_2MOD):
    n     = len(indices)
    label = '_'.join(f'{m}{t}' for m, t in combos)
    cfile = os.path.join(CACHE_DIR, f'cams_{label}_{condition}_n{n}.npz')
    keys  = [f'cam_{m}_{t}' for m, t in combos]

    if os.path.exists(cfile):
        data = np.load(cfile, allow_pickle=True)
        print(f'Loaded cache {os.path.basename(cfile)}')
        return {k: data[k] for k in keys}, np.array(data['indices'])

    imgs = attacked_imgs if condition == 'uni_attack' else clean_224
    imgs = [imgs[i] for i in indices]

    cam_lists = {k: [] for k in keys}
    t0 = time.time()
    for j, img in enumerate(imgs):
        for (ml, tl), k in zip(combos, keys):
            emb      = TEXT_EMBS[(ml, tl)]
            fn       = gradcam_en_with_emb if ml == 'en' else gradcam_zh_with_emb
            cam, _   = fn(img, emb)
            cam_lists[k].append(cam)
        if (j + 1) % 50 == 0:
            print(f'  CAM {j+1}/{n} [{time.time()-t0:.1f}s]')

    data_save = {k: np.stack(cam_lists[k]) for k in keys}
    data_save['indices'] = np.array(indices)
    np.savez(cfile, **data_save)
    print(f'Saved cache {os.path.basename(cfile)} [{time.time()-t0:.1f}s]')
    return {k: data_save[k] for k in keys}, np.array(indices)

print('CAM cache helper ready (unilingual 2-mod).')

CAM cache helper ready (unilingual 2-mod).


## 7. Threshold sweep

In [14]:
THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
print('Computing tune-set CAMs...')
cd_tune, _ = compute_and_cache_cams('uni_attack', tune_idx)

sweep_rows = []
imgs_tune  = [attacked_imgs[i] for i in tune_idx]
base = {ml: float((preds_attacked[ml][tune_idx] == true[tune_idx]).mean()) for ml in LANGS}
for thr in THRESHOLDS:
    sal_list = [n_cam_intersection(*[cd_tune[f'cam_{m}_{t}'][j] for m, t in COMBOS_UNI_2MOD])
                for j in range(len(tune_idx))]
    masks  = [cam_to_mask(s, thr, 3) for s in sal_list]
    masked = [apply_mask(imgs_tune[j], masks[j]) for j in range(len(tune_idx))]
    for ml in LANGS:
        p = classify(models[ml], masked, CLASSES[ml])
        sweep_rows.append({
            'model': ml, 'threshold': thr,
            'baseline_acc': base[ml],
            'masked_acc':   float((p == true[tune_idx]).mean()),
            'masked_asr':   float((p == target[tune_idx]).mean()),
        })

en_rows    = [r for r in sweep_rows if r['model'] == 'en']
best_row   = max(en_rows, key=lambda r: r['masked_acc'])
BEST_THR   = best_row['threshold']
print(f'Best threshold: {BEST_THR}  '
      f'{100*best_row["baseline_acc"]:.1f}% -> {100*best_row["masked_acc"]:.1f}%')

Computing tune-set CAMs...
  CAM 50/100 [3.4s]
  CAM 100/100 [6.6s]
Saved cache cams_enen_zhen_uni_attack_n100.npz [6.6s]
Best threshold: 0.85  3.0% -> 24.0%


## 8. Full evaluation (1000 images)

In [15]:
print('Computing full CAM cache...')
cd_full, _       = compute_and_cache_cams('uni_attack', all_idx)
cd_clean_full, _ = compute_and_cache_cams('clean',      all_idx)

def make_sal_list(cam_data, combos, n):
    return [n_cam_intersection(*[cam_data[f'cam_{m}_{t}'][j] for m, t in combos])
            for j in range(n)]

sal_list     = make_sal_list(cd_full, COMBOS_UNI_2MOD, len(all_idx))
masks        = [cam_to_mask(s, BEST_THR, 3) for s in sal_list]
masked_imgs  = [apply_mask(attacked_imgs[j], masks[j]) for j in range(len(all_idx))]

defense_res  = {}
for ml in LANGS:
    base_p = preds_attacked[ml]
    new_p  = classify(models[ml], masked_imgs, CLASSES[ml])
    wrong  = base_p != true
    defense_res[ml] = {
        'acc':           float((new_p == true).mean()),
        'asr':           float((new_p == target).mean()),
        'recovery_rate': float((wrong & (new_p == true)).sum() / wrong.sum()) if wrong.any() else 0.0,
        'baseline_acc':  float((base_p == true).mean()),
        'baseline_asr':  float((base_p == target).mean()),
    }

# Clean degradation
sal_clean   = make_sal_list(cd_clean_full, COMBOS_UNI_2MOD, len(all_idx))
masks_clean = [cam_to_mask(s, BEST_THR, 3) for s in sal_clean]
m_clean     = [apply_mask(clean_224[j], masks_clean[j]) for j in range(len(all_idx))]
clean_deg   = {}
for ml in LANGS:
    cp = classify(models[ml], m_clean, CLASSES[ml])
    clean_deg[ml] = {
        'baseline_acc': clean_acc[ml],
        'masked_acc':   float((cp == true).mean()),
        'delta_acc':    float((cp == true).mean()) - clean_acc[ml],
    }

results = {
    'setup':             'unilingual',
    'method':            'cam_2mod',
    'attack':            'unilingual',
    'n_images':          len(all_idx),
    'best_threshold':    BEST_THR,
    'combos':            str(COMBOS_UNI_2MOD),
    'inference_cost':    6,
    'clean_acc':         clean_acc,
    'baseline_acc':      {ml: defense_res[ml]['baseline_acc'] for ml in LANGS},
    'baseline_asr':      {ml: defense_res[ml]['baseline_asr'] for ml in LANGS},
    'defense':           defense_res,
    'clean_degradation': clean_deg,
    'coverage_mean':     float(np.mean([m.mean() for m in masks])),
    'defense_acc_mean':  float(np.mean([defense_res[ml]['acc'] for ml in LANGS])),
    'defense_asr_mean':  float(np.mean([defense_res[ml]['asr'] for ml in LANGS])),
}
out_path = f'{RESULTS_DIR}/confusion_results_cam_defense.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'Saved -> {out_path}')
print('\nDefence summary:')
for ml in LANGS:
    d = defense_res[ml]
    print(f'  model_{ml}: {100*d["baseline_acc"]:.1f}% -> {100*d["acc"]:.1f}%  '
          f'ASR {100*d["baseline_asr"]:.1f}% -> {100*d["asr"]:.1f}%')

Computing full CAM cache...
  CAM 50/1000 [3.2s]
  CAM 100/1000 [6.5s]
  CAM 150/1000 [9.8s]
  CAM 200/1000 [13.1s]
  CAM 250/1000 [16.3s]
  CAM 300/1000 [19.5s]
  CAM 350/1000 [22.6s]
  CAM 400/1000 [25.8s]
  CAM 450/1000 [29.1s]
  CAM 500/1000 [32.3s]
  CAM 550/1000 [35.6s]
  CAM 600/1000 [38.9s]
  CAM 650/1000 [42.1s]
  CAM 700/1000 [45.2s]
  CAM 750/1000 [48.5s]
  CAM 800/1000 [51.7s]
  CAM 850/1000 [54.9s]
  CAM 900/1000 [58.0s]
  CAM 950/1000 [61.3s]
  CAM 1000/1000 [64.5s]
Saved cache cams_enen_zhen_uni_attack_n1000.npz [64.5s]
  CAM 50/1000 [3.2s]
  CAM 100/1000 [6.4s]
  CAM 150/1000 [9.6s]
  CAM 200/1000 [12.8s]
  CAM 250/1000 [16.4s]
  CAM 300/1000 [19.6s]
  CAM 350/1000 [22.7s]
  CAM 400/1000 [25.9s]
  CAM 450/1000 [28.9s]
  CAM 500/1000 [32.2s]
  CAM 550/1000 [35.4s]
  CAM 600/1000 [38.6s]
  CAM 650/1000 [41.8s]
  CAM 700/1000 [45.0s]
  CAM 750/1000 [48.2s]
  CAM 800/1000 [51.3s]
  CAM 850/1000 [54.4s]
  CAM 900/1000 [57.6s]
  CAM 950/1000 [60.8s]
  CAM 1000/1000 [63.9s]
Sa

## 9. Visual diagnostics

In [16]:
fig, ax = plt.subplots(figsize=(4, 3))
vals = np.array([[defense_res[ml]['acc'] - defense_res[ml]['baseline_acc']]
                  for ml in LANGS]) * 100
im = ax.imshow(vals, vmin=-20, vmax=60, cmap='RdYlGn')
ax.set_xticks([0]); ax.set_xticklabels(['unilingual EN attack'])
ax.set_yticks(range(len(LANGS))); ax.set_yticklabels([f'model_{l}' for l in LANGS])
ax.set_title('Accuracy delta after 2-mod CAM masking (pp)')
for i, ml in enumerate(LANGS):
    ax.text(0, i, f'{vals[i,0]:+.1f}', ha='center', va='center', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, format=lambda x, _: f'{x:+.0f}pp')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/accuracy_delta_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved -> accuracy_delta_matrix.png')

Saved -> accuracy_delta_matrix.png
